In [28]:
from utils import getDevice, collate_fn #important to impot at the start for reproducibility :3 

In [29]:
# GENERAL IMPORTS
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np
import time
import torch
import gc
import os
import pandas as pd
from IPython.display import clear_output

from torchvision.models import vgg16
from torchvision import models
from torchvision import transforms
from torchvision.ops import roi_pool
import torchvision
from torch import nn
from torch.utils.data import Dataset, DataLoader, random_split

# CUSTIM FUNCTIONS AND VARIABLES
from kitty import depth_read, listPicsWith, togglePath, MatchDepthToCar, KITTY_PATH
from yolo import getEmbedFromResults, getCropsFromResults, getCropsFromResult, getEmbedFromCrops, CLASSES_YOLO, CONFIDENCE_YOLO

# YOLO STUFF
from ultralytics import YOLO

#DINO STUFF
from transformers import AutoImageProcessor, AutoModel
from transformers.image_utils import load_image

In [30]:
DEVICE = getDevice()
HEIGHT = 376
WIDTH = 1242
TARGET_TYPES = ['Car']
BATCH_SIZE = 4
NUM_WORKERS = 4

In [31]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((HEIGHT, WIDTH))
])

#DATASET IS LOADED AND DATALOADER IS CREATED

kitty_dataset =torchvision.datasets.Kitti(root="../datasets/", train=True,transform=transform, download=True)
kittty_dataloader = torch.utils.data.DataLoader(
    kitty_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=True, 
    collate_fn=collate_fn, 
    num_workers=NUM_WORKERS,
    pin_memory=True
)

KITTY_LENGTH = len(kitty_dataset)
BATCH_COUNT = KITTY_LENGTH // BATCH_SIZE + (KITTY_LENGTH % BATCH_SIZE > 0)
print(f"KITTY dataset is length: {KITTY_LENGTH}\nDataloader has length: {len(kittty_dataloader)}\nBatch count {BATCH_COUNT}")

#I try a single batch to analyze the data structure and types
images, targets_batch = next(iter(kittty_dataloader))
print(f"Type of the images data is: {type(images)} type of image {type(images[0])} len: {len(images)} and type of the target is {type(targets_batch)} of len {len(targets_batch)}")

KITTY dataset is length: 7481
Dataloader has length: 1871
Batch count 1871
[2026-04-18 21:58:13.729] [warning] [sycl_collector.h:388] Another subscriber already subscribed to Sycl runtime events, so PTI will not subscribe to them. It will affect correctness of PTI profile: e.g. report zero XPU time for CPU callers of GPU kernels.
Using device: xpu :3
Setting seed to 42:3
[2026-04-18 21:58:14.737] [warning] [sycl_collector.h:388] Another subscriber already subscribed to Sycl runtime events, so PTI will not subscribe to them. It will affect correctness of PTI profile: e.g. report zero XPU time for CPU callers of GPU kernels.
Using device: xpu :3
Setting seed to 42:3
[2026-04-18 21:58:15.744] [warning] [sycl_collector.h:388] Another subscriber already subscribed to Sycl runtime events, so PTI will not subscribe to them. It will affect correctness of PTI profile: e.g. report zero XPU time for CPU callers of GPU kernels.
Using device: xpu :3
Setting seed to 42:3
[2026-04-18 21:58:16.740] [w

In [32]:
for i, targets in enumerate(targets_batch):
    print(f"Target type is {type(targets)} and length is {len(targets)}")
    for target in targets:
        print(type(target))  # Should be a dict
        obj_types  = target["type"]        # e.g. ['Car', 'Pedestrian']
        locations  = target["location"]    # Tensor [N, 3]: (X, Y, Z) in metres
        bboxes     = torch.Tensor(target["bbox"])        # Tensor [N, 4]: (x1, y1, x2, y2) pixels
        print(f"Object types: {obj_types} of type {type(obj_types)}")
        print(f"Locations: {locations} of type {type(locations)}")
        print(f"Bounding boxes: {bboxes} of type {type(bboxes)}")

    print("")

Target type is <class 'list'> and length is 7
<class 'dict'>
Object types: Car of type <class 'str'>
Locations: [-19.78, 1.76, 22.06] of type <class 'list'>
Bounding boxes: tensor([  0.0000, 180.0200,  41.1700, 229.2000]) of type <class 'torch.Tensor'>
<class 'dict'>
Object types: Car of type <class 'str'>
Locations: [-3.89, 1.81, 34.52] of type <class 'list'>
Bounding boxes: tensor([505.7300, 177.3000, 550.4900, 212.7600]) of type <class 'torch.Tensor'>
<class 'dict'>
Object types: Car of type <class 'str'>
Locations: [-3.77, 2.06, 55.75] of type <class 'list'>
Bounding boxes: tensor([549.5800, 179.6300, 572.5200, 200.2600]) of type <class 'torch.Tensor'>
<class 'dict'>
Object types: Car of type <class 'str'>
Locations: [0.02, 1.77, 28.72] of type <class 'list'>
Bounding boxes: tensor([589.1500, 176.0200, 634.0000, 220.7000]) of type <class 'torch.Tensor'>
<class 'dict'>
Object types: DontCare of type <class 'str'>
Locations: [-1000.0, -1000.0, -1000.0] of type <class 'list'>
Bounding

In [33]:
yolo_embeds_paths =[]
dino_embeds_paths =[]

In [34]:
DINO_MODEL_ID = "facebook/dinov3-vits16-pretrain-lvd1689m"

image_processor = AutoImageProcessor.from_pretrained(DINO_MODEL_ID)
dinov3 = AutoModel.from_pretrained(
    DINO_MODEL_ID,
    dtype=torch.float16,
    device_map="auto"
).to(DEVICE)

def DINOgetEmbed(image):
    """
    Get DINOv3 embeddings from an image or a list of images. 
    Args:
        image: string path or PIL image or list of ready to pricess opened imaged
    """
    #case 1 - image is a path
    if type(image) == str:
        image = load_image(image)
    
    #case 2 is pil image already or compatible with the class
    inputs = image_processor(images=image, return_tensors="pt").to(DEVICE)

    #retrieve embeddings
    dinov3.eval()
    with torch.inference_mode():
        outputs = dinov3(**inputs)

    return outputs.pooler_output  # final embeddings

Loading weights: 100%|██████████| 211/211 [00:00<00:00, 6276.36it/s]


In [35]:


def extract_crops_depth(images_batch, targets_batch):
    # 1 IMAGES
        stacked_images = torch.stack(images_batch)
        # 2,3 BOXES and DEPTHS
        boxes =[]
        depths = []
        skip = []
        image_indicies = set(range(len(images_batch)))
        keep=[]
        crops=[]
        for i, targets in enumerate(targets_batch):
            boxes_with_target = [torch.tensor(target["bbox"], dtype=torch.int32) for target in targets if target["type"] in TARGET_TYPES]
            depths_with_target = [target["location"][2] for target in targets if target["type"] in TARGET_TYPES]
            if len(boxes_with_target) > 0:
                boxes.append(torch.stack(boxes_with_target))
                depths.append(torch.Tensor(depths_with_target))
            else:
                skip.append(i)

        # 3.5 check that there is actually data 
        if len(boxes) == 0:
            print("No boxes found in this batch, skipping...")
            del boxes, images_batch, targets_batch
            gc.collect()
            return -1, -1

        #3.6 skip images that we do not want and add the depths
        keep = list(image_indicies - set(skip))
        stacked_images = stacked_images[keep,:,:,:]
        depths = torch.cat(depths).cpu()
        #depths_true_all.extend(depths.tolist())

        # 4 crop the images to a specific size
        for i in range(stacked_images.shape[0]):
            # the current data is fetched
            image = stacked_images[i]
            boxes_cur = boxes[i]

            #images are copped to the box
            for box in boxes_cur:
                x1,y1,x2,y2 = box.tolist()
                crop_tensor = image[:, y1:y2, x1:x2]
                crop_np = crop_tensor.permute(1, 2, 0).numpy()
                crop_np = (crop_np * 255).clip(0, 255).astype(np.uint8)
                crops.append(crop_np)

        #exit function and clear the local variables
        del stacked_images, boxes, targets_batch
        return crops, depths.tolist()

def extract_crops(loader: DataLoader):
    """
    Extract the crops before the embeddings are retrieved.

    Args:
        loader: dataloader for the dataset
    """
    depths_true_all = []
    counter = 0
    chunk_yolo=[]
    chunk_dino=[]
    chunk_count=0
    
    for images_batch, targets_batch in loader:
        #preporcess the data, format it in a way that is expected by the model===================
        clear_output(wait=True)
        print(f"Batch {counter} in progress; ({counter/len(loader)*100:.2f}%);")

        #extract boxes and depths to be added and saved==========================================
        crops, depths = extract_crops_depth(images_batch, targets_batch)
        if crops==-1 and depths==-1:
            continue
        depths_true_all.extend(depths)

        #YOLO embeds=============================================================================
        #embeds = []
        for crop in crops:
            em_yolo = getEmbedFromCrops(crop)
            em_dino = DINOgetEmbed(crop)
            #print(f"the type of em dino is {type(em_dino)} of shape{em_dino.shape} the item added is {em_dino[0].shape}")
            chunk_yolo.append(em_yolo[0])
            chunk_dino.append(em_dino[0])

        #flush the batch into the memory
        chumky_path_yolo = f"../data/yolo_embeds_chunk_{chunk_count}.pt"
        chumky_path_dino = f"../data/dino_embeds_chunk_{chunk_count}.pt"
        torch.save(torch.stack(chunk_yolo), chumky_path_yolo)
        torch.save(torch.stack(chunk_dino), chumky_path_dino)
        #save the path
        yolo_embeds_paths.append(chumky_path_yolo)   
        dino_embeds_paths.append(chumky_path_dino)     

        #clear the variables and incremenet the counter===========================================
        del crops, depths, chunk_yolo, chunk_dino
        chunk_yolo = []
        chunk_dino = []
        chunk_count += 1
        counter+=1

    gc.collect() 
    return depths_true_all

In [36]:
depths_true = extract_crops(kittty_dataloader)

Batch 1870 in progress; (99.95%);





In [37]:
print("Merging DINO embedding chunks...")
all_chunks = [torch.load(p) for p in dino_embeds_paths]
full_embeds = torch.cat(all_chunks, dim=0)
torch.save(full_embeds, "../data/embeds_dino_og_boxes.pt")
print(f"Saved DINO embeddings: {full_embeds.shape}")
del all_chunks, full_embeds
gc.collect()

for path in dino_embeds_paths:
    os.remove(path)

Merging DINO embedding chunks...
Saved DINO embeddings: torch.Size([28742, 384])


In [38]:
print("Merging YOLO embedding chunks...")
all_chunks = [torch.load(p) for p in yolo_embeds_paths]
full_embeds = torch.cat(all_chunks, dim=0)
torch.save(full_embeds, "../data/embeds_yolo_og_boxes.pt")
print(f"Saved YOLO embeddings: {full_embeds.shape}")
del all_chunks, full_embeds
gc.collect()

for path in yolo_embeds_paths:
    os.remove(path)

Merging YOLO embedding chunks...
Saved YOLO embeddings: torch.Size([28742, 256])


In [39]:
depths_tensor = torch.Tensor(depths_true)
torch.save(depths_tensor, "../data/true_depth.pt")
print(f"Saved the true depths into a file with shape: {depths_tensor.shape}")
del depths_tensor
gc.collect()

Saved the true depths into a file with shape: torch.Size([28742])


18